# Predict 1-Year US Stock Returns from Fundamentals

**Task:** Regression — predict `return_pct`, the 1-year forward price return for a US stock, using only publicly available fundamental financial metrics recorded at the start of each period.

---

| | Train | Test |
|---|---|---|
| Rows | 23,070 | 8,520 |
| Features | 32 fundamental metrics + metadata | same (no target) |
| Unique tickers | 1,734 | — |
| Time coverage | 2019–2022 (quarterly windows) | — |

**Target distribution at a glance:**
- Range: −99% to +10,571% &nbsp;|&nbsp; Skew ≈ 38 (extreme right tail)
- Median: ~3.5% &nbsp;|&nbsp; Mean: ~18.8%
- ~46% of stocks had negative 1-year returns

**Key challenges to keep in mind throughout:**
1. **Extreme target skew** — a handful of stocks (meme stocks, penny stocks) have thousands-of-percent returns that will distort loss functions
2. **Heavy missing data** — several columns are up to 93% missing; strategy and order of operations matters
3. **Extreme feature outliers** — financial ratios contain erroneous extremes (e.g. `pe_ttm` max = 1.25 billion)
4. **Temporal panel structure** — 1,734 tickers × ~4 quarterly snapshots × 4 years; naive K-fold leaks future into training

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr

# Modeling
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import lightgbm as lgb  # pip install lightgbm

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

---
## Section 1 — Data Loading & Initial Overview

In [3]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
sub   = pd.read_csv('../data/sample_submission.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head()

Train: (23070, 39)  |  Test: (8520, 36)


,id,ticker,start_year,period_start,period_end,return_pct,pe_ttm,price_to_book,price_to_sales,growth_pe_ratio,gross_margin,operating_margin,net_margin,roa,roe,rote,revenue_growth_3y,revenue_growth_yoy,revenue_ttm,net_income_ttm,income_before_tax,eps_basic,eps_diluted,total_assets,stockholders_equity,current_assets,current_liabilities,long_term_debt,goodwill,inventory,current_ratio,quick_ratio,debt_to_equity,dividend_yield,dividends_ttm,dividends_paid_ttm,shares_outstanding,shares_diluted,sector_code
0,0,AAPL,2022,2022-03-26,2023-04-01,-5.62,25.28,38.23,6.68,1.95,43.75,30.82,26.41,29.07,151.24,151.24,49.34,18.63,3.860170e+11,1.019350e+11,3.013900e+10,1.54,1.52,3.506620e+11,6.739900e+10,1.181800e+11,1.275080e+11,2.066460e+11,0.0,5.460000e+09,0.93,0.88,3.07,0.57,1.473400e+10,1.468700e+10,1.620757e+10,1.640332e+10,0.0
1,1,AAPL,2022,2022-06-25,2023-07-01,36.93,20.97,35.95,5.39,2.37,43.26,27.82,25.71,29.63,171.46,171.46,49.61,11.63,3.875420e+11,9.963300e+10,2.306600e+10,1.20,1.20,3.363090e+11,5.810700e+10,1.122920e+11,1.298730e+11,1.894000e+11,0.0,5.433000e+09,0.86,0.82,3.26,0.71,1.477800e+10,1.473400e+10,1.609538e+10,1.626220e+10,0.0
2,2,AAPL,2022,2022-09-24,2023-09-30,13.81,22.23,43.78,5.63,2.32,43.31,30.29,25.31,28.29,196.96,196.96,51.56,7.79,3.943280e+11,9.980300e+10,1.191030e+11,6.15,6.11,3.527550e+11,5.067200e+10,1.354050e+11,1.539820e+11,1.979180e+11,0.0,4.946000e+09,0.88,0.85,3.91,0.67,1.484100e+10,1.479300e+10,1.594342e+10,1.632582e+10,0.0
3,3,AAPL,2022,2022-12-31,2023-12-30,48.18,20.13,33.78,4.94,2.22,42.96,30.74,24.56,27.45,167.77,167.77,44.77,2.44,3.875370e+11,9.517100e+10,3.562300e+10,1.89,1.88,3.467470e+11,5.672700e+10,1.287770e+11,1.372860e+11,1.992540e+11,0.0,6.820000e+09,0.94,0.89,3.51,0.78,1.487700e+10,1.484000e+10,1.584241e+10,1.595572e+10,0.0
4,4,AAPL,2021,2021-03-27,2022-03-26,44.15,23.43,25.84,5.49,1.35,42.51,30.70,23.45,22.63,110.31,110.31,31.52,21.43,3.254060e+11,7.631100e+10,2.801100e+10,1.41,1.40,3.371580e+11,6.917800e+10,1.214650e+11,1.063850e+11,2.172840e+11,0.0,5.219000e+09,1.14,1.09,3.14,0.80,1.422700e+10,1.421200e+10,1.668630e+10,1.692916e+10,0.0


In [ ]:
#Overview of the data
train.describe()

,id,start_year,return_pct,pe_ttm,price_to_book,price_to_sales,growth_pe_ratio,gross_margin,operating_margin,net_margin,roa,roe,rote,revenue_growth_3y,revenue_growth_yoy,revenue_ttm,net_income_ttm,income_before_tax,eps_basic,eps_diluted,total_assets,stockholders_equity,current_assets,current_liabilities,long_term_debt,goodwill,inventory,current_ratio,quick_ratio,debt_to_equity,dividend_yield,dividends_ttm,dividends_paid_ttm,shares_outstanding,shares_diluted,sector_code
count,23070.000000,23070.000000,23070.000000,2.069400e+04,1.753700e+04,19357.000000,15054.000000,8660.000000,1.935100e+04,20379.000000,18387.000000,1.811600e+04,1.861100e+04,16020.000000,18527.000000,2.048000e+04,2.184100e+04,1.508500e+04,2.241600e+04,2.210700e+04,2.019400e+04,2.003900e+04,1.602200e+04,1.600800e+04,1.543400e+04,1.604000e+04,1.155600e+04,15676.000000,15676.000000,14478.000000,6453.000000,6.734000e+03,1.565000e+03,1.490900e+04,1.818300e+04,22966.000000
mean,11534.500000,2020.620156,18.778736,4.419552e+06,1.318639e+03,38.474639,0.812582,64.951527,-4.193572e+02,-134.911569,2.387002,1.205656e+03,3.908105e+02,217.217064,63.734911,1.134252e+10,1.020397e+09,5.926303e+08,1.474358e+02,2.239556e+02,4.046011e+10,7.450112e+09,5.468800e+09,4.208311e+09,1.034960e+10,3.666587e+09,1.330403e+09,2.893604,2.545415,31.808600,5.989950,5.273457e+08,3.811204e+08,3.014210e+08,3.480357e+08,3.137159
std,6659.879691,1.116563,138.650112,1.848069e+08,5.618679e+04,623.685568,14.546072,2091.963498,1.270640e+04,4522.532183,114.604440,4.354509e+04,3.391791e+04,6387.684648,3137.563635,3.366094e+10,4.487041e+09,3.474396e+09,1.270971e+04,1.693667e+04,2.124295e+11,2.456240e+10,1.470851e+10,1.202842e+10,3.365260e+10,9.808444e+09,4.310532e+09,5.969118,5.976915,1036.152788,132.831327,1.592582e+09,1.664419e+09,1.867548e+09,2.601358e+09,2.609367
min,0.000000,2019.000000,-99.170000,-2.378991e+08,-1.393592e+05,0.000000,-100.000000,-82762.500000,-1.237600e+06,-218214.100000,-2617.510000,-1.206824e+06,-1.206824e+06,-99.980000,-99.970000,0.000000e+00,-2.281900e+10,-6.304900e+10,-3.110000e+03,-3.110000e+03,-1.469790e+08,-2.712300e+10,-5.754500e+07,-6.904500e+10,-1.220600e+10,-1.256200e+10,-3.700000e+06,-2.110000,-1.640000,-12453.560000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,5767.250000,2020.000000,-19.787500,-3.400000e-01,1.440000e+00,1.060000,0.000000,25.117500,1.390000e+00,0.170000,0.090000,-1.000000e-01,-1.286500e+01,4.377500,-0.455000,6.479320e+08,-4.000000e+00,-1.073000e+06,0.000000e+00,0.000000e+00,1.677407e+09,3.588335e+08,4.773508e+08,1.977395e+08,4.800000e+07,1.389700e+07,3.628494e+07,1.150000,0.920000,0.010000,0.000000,0.000000e+00,0.000000e+00,4.496902e+07,4.447116e+07,1.000000
50%,11534.500000,2021.000000,3.500000,1.521000e+01,2.960000e+00,2.650000,0.690000,38.605000,1.022000e+01,7.270000,3.450000,1.025000e+01,9.700000e+00,23.465000,8.200000,2.482696e+09,1.399830e+08,6.252400e+07,6.300000e-01,6.200000e-01,5.643240e+09,1.462366e+09,1.422276e+09,7.686850e+08,1.561027e+09,4.212110e+08,2.802000e+08,1.700000,1.350000,0.650000,0.680000,2.761000e+07,0.000000e+00,9.750000e+07,9.832232e+07,3.000000
75%,17301.750000,2022.000000,33.827500,3.042750e+01,6.610000e+00,6.350000,2.050000,56.525000,2.050500e+01,16.040000,7.880000,2.120000e+01,2.580500e+01,53.850000,21.590000,8.571423e+09,7.032130e+08,3.370000e+08,1.780000e+00,1.760000e+00,1.926578e+10,5.487192e+09,4.083750e+09,2.941149e+09,7.241221e+09,2.694797e+09,1.056998e+09,2.840000,2.290000,1.570000,2.470000,2.903675e+08,2.237000e+08,2.382685e+08,2.550000e+08,5.000000
max,23069.000000,2022.000000,10571.110000,1.251553e+10,5.271275e+06,53118.760000,100.000000,137639.680000,9.353684e+04,104247.690000,14705.600000,3.361580e+06,3.361580e+06,679707.000000,414794.440000,6.001120e+11,1.057770e+11,1.191030e+11,1.740000e+06,1.730000e+06,4.305288e+12,5.081410e+11,1.881430e+11,1.694220e+11,5.602340e+11,1.466620e+11,8.696100e+10,211.520000,211.520000,69786.500000,7310.330000,2.008700e+10,2.434000e+10,1.217753e+11,1.751079e

### Column Glossary

The 32 feature columns fall into 6 financial categories:

| Category | Columns | What they measure |
|---|---|---|
| **Valuation** | `pe_ttm`, `price_to_book`, `price_to_sales`, `growth_pe_ratio` | How expensive the stock is relative to earnings/assets/sales |
| **Profitability** | `gross_margin`, `operating_margin`, `net_margin`, `roa`, `roe`, `rote` | How efficiently the company converts revenue to profit |
| **Growth** | `revenue_growth_yoy`, `revenue_growth_3y` | Revenue momentum |
| **Income statement** | `revenue_ttm`, `net_income_ttm`, `income_before_tax`, `eps_basic`, `eps_diluted` | Absolute size of earnings (trailing twelve months) |
| **Balance sheet** | `total_assets`, `stockholders_equity`, `current_assets`, `current_liabilities`, `long_term_debt`, `goodwill`, `inventory` | Financial structure and health |
| **Liquidity & leverage** | `current_ratio`, `quick_ratio`, `debt_to_equity` | Ability to meet short-term obligations |
| **Dividends & shares** | `dividend_yield`, `dividends_ttm`, `dividends_paid_ttm`, `shares_outstanding`, `shares_diluted` | Capital return and dilution |
| **Metadata** | `ticker`, `start_year`, `period_start`, `period_end`, `sector_code` | Identity and time |

**Note:** `period_start` and `period_end` are present in train but **not** in test — do not use them as features directly.

---
## Section 2 — Exploratory Data Analysis

Objective: understand the distribution of the target and features, identify data quality issues, and build intuition for which fundamentals may predict 1-year returns.

---

### 2.1 Target Variable — `return_pct`

**What to plot:**
- Histogram of raw `return_pct` — expect extreme right skew (skew ≈ 38)
- Histogram after clipping extreme outliers (e.g. cap at 300%) to see the bulk of the distribution
- Box plot by `start_year` — 2020 is the COVID crash/recovery; 2022 is the rate-hike year
- Distribution by `sector_code`

**Key facts:**
- Range: −99.17% to +10,571% &nbsp;|&nbsp; Median ≈ 3.5% &nbsp;|&nbsp; Mean ≈ 18.8%
- ~46% of stocks had negative 1-year returns — this is a genuinely hard regression problem, not just ranking
- Extreme outlier stocks (penny stocks, meme stocks, small-cap recoveries) will dominate RMSE

**Decision to make after plotting:** should we clip or log-transform the target for training? (addressed in Section 4.1)

In [ ]:
# 2.1 — Target variable analysis


### 2.2 Missing Values Analysis

**What to compute and plot:**
- Count and % missing per column, sorted descending — plot as a horizontal bar chart
- Group features by severity:

| Severity | Threshold | Columns | Recommended action |
|---|---|---|---|
| Critical | > 70% missing | `dividends_paid_ttm` (93%), `dividend_yield` (72%) | Drop or binary flag |
| High | 40–70% | `gross_margin` (62%), `growth_pe_ratio` (35%), `inventory` (50%) | Binary flag + impute |
| Moderate | 20–40% | Most balance sheet items | Median impute by sector |
| Low | < 20% | `eps_basic`, `net_income_ttm`, etc. | Median impute globally |

**Key insight:** missing `dividend_yield` is **informative** — it means the company pays no dividend (typically a growth stock). A `has_dividend` binary feature captures this signal before imputing the column to 0.

In [ ]:
# 2.2 — Missing values bar chart; group by severity tier


### 2.3 Feature Distributions & Outliers

**Valuation ratios** (`pe_ttm`, `price_to_book`, `price_to_sales`, `growth_pe_ratio`):
- Severe outliers: `pe_ttm` max = 1.25 billion; `price_to_book` max = 5.27 million — these are errors
- Negative values are **valid** (companies with losses or negative equity) — do not zero-floor them
- Plot histograms after clipping to the 1st–99th percentile range to see the real distribution shape

**Profitability metrics** (`gross_margin`, `operating_margin`, `net_margin`, `roa`, `roe`, `rote`):
- Margins are percentages; typical healthy ranges: gross 20–80%, operating 5–30%, net 2–20%
- `roe` can be very large for companies with extensive buybacks (minimal equity base) — valid but extreme

**Growth metrics** (`revenue_growth_yoy`, `revenue_growth_3y`):
- Centered around 0–20% for mature companies; extreme values for early-stage growth firms

**Balance sheet items** (`total_assets`, `revenue_ttm`, etc.):
- Absolute dollar values spanning many orders of magnitude (small-cap to mega-cap)
- Tree models handle this natively; for linear models, apply `np.log1p` to all positive-dollar columns

**Recommended plots:** 3×4 grid of histograms (one per feature group), showing the winsorized distribution with the mean and median marked.

In [ ]:
# 2.3 — Log-scale histograms of valuation ratios; distributions of profitability and growth metrics


### 2.4 Temporal Analysis — Returns by Year

`start_year` covers 2019–2022, each representing a distinct market regime:

| Year | Market context | Expected return profile |
|---|---|---|
| 2019 | Strong bull market | Broadly positive; growth stocks outperform |
| 2020 | COVID crash (Q1) then dramatic V-shaped recovery | High variance; timing matters enormously |
| 2021 | Post-COVID boom; growth stocks at peak valuations | High median returns; speculation |
| 2022 | Rate-hike cycle begins; growth stocks collapse | Negative median; value outperforms |

**What to plot:**
- Violin or box plot of `return_pct` (clipped) by `start_year`
- Median and mean return per year (table or line chart)
- Fraction of stocks with positive returns per year

**Modeling implication:** training on all 4 years together teaches the model to predict the *average* across very different macro environments. Since the test data covers 2023+, the most recent regime (2022, rate hikes) may be most representative. Consider whether `start_year` as a feature helps or hurts generalization.

In [ ]:
# 2.4 — Distribution of return_pct by start_year (violin/box); fraction of positive returns per year


### 2.5 Sector Analysis

There are 11 sector codes (0–10). Based on the sample data, sector 0 appears to be Technology (AAPL, etc.). The dataset doesn't include sector names but sectors can be reverse-engineered from tickers.

**What to plot:**
- Box plot of `return_pct` (after clipping to e.g. ±200%) by `sector_code` — some sectors are structurally more volatile
- Count of unique tickers per sector — check for class imbalance (sector 10 has only 381 samples vs. 3,840 for sector 1)
- Heatmap of mean feature values by sector (normalized) — sectors differ structurally: banks have very high debt-to-equity, tech has very high P/E, utilities have high dividend yields

**Key question:** does sector predict return level, or mainly predict feature *ranges*?
- If sectors differ primarily in feature distributions (not return distributions), `sector_code` is most useful as a grouping variable for imputation
- If some sectors systematically outperform, `sector_code` is a direct feature

In [ ]:
# 2.5 — Return distribution by sector; mean feature values by sector (heatmap)


### 2.6 Correlation Analysis

**Feature–target correlations:**
- Compute the Spearman correlation of each feature with `return_pct` (use Spearman, not Pearson — it handles the non-normal distribution better)
- Plot as a horizontal sorted bar chart, colored by sign
- Spearman is also the right metric here because for portfolio use cases we care about *ranking* stocks, not predicting exact values

**Inter-feature correlations:**
- Plot a heatmap of the full correlation matrix for all numeric features
- Identify **redundant pairs** to drop before modeling: `eps_basic` vs `eps_diluted` (~0.99), `dividends_ttm` vs `dividends_paid_ttm`, `current_ratio` vs `quick_ratio`
- Identify **information clusters**: the valuation group (P/E, P/B, P/S), profitability group (margins, ROE, ROA), balance-sheet size group (assets, revenue, equity)

**Expected finding:** individual feature–target correlations will be weak (|ρ| < 0.10 for most). This is normal for stock return prediction — the predictive signal comes from combinations of features, not single ratios.

**Action:** flag any feature with |ρ| < 0.02 as a candidate for removal (near-zero marginal contribution).

In [ ]:
# 2.6 — Spearman correlation of each feature with return_pct (bar chart); inter-feature heatmap


---
## Section 3 — Key Insights Summary

*Fill in after completing Section 2. Use this as a decision log before moving to modeling.*

**Target variable:**
- [ ] Describe the shape of the distribution and whether transformation is needed
- [ ] Which year had the best/worst median returns? How extreme is 2020?
- [ ] Which sectors are most and least volatile?

**Data quality:**
- [ ] Which columns will be dropped and why?
- [ ] Which features have suspicious outlier values (beyond data noise)?
- [ ] Chosen imputation strategy for each severity group

**Predictive signal:**
- [ ] Top 5 features most correlated (Spearman) with `return_pct`
- [ ] Features with correlation near zero — candidates for removal
- [ ] Any feature groups that appear redundant (high inter-correlation)?

**Modeling implications:**
- [ ] Target transformation decision (clip / log / none) and rationale
- [ ] Should `start_year` be included as a feature or only used for CV grouping?
- [ ] Is there enough signal for a single global model, or should sector-specific models be explored?

> **Baseline expectation:** individual Spearman correlations for stock return prediction are typically in the 0.03–0.10 range. A model achieving ρ > 0.10 is a strong result for this problem.

---
## Section 4 — Feature Engineering & Preprocessing

All preprocessing steps follow a strict rule: **fit on training data, apply to test.** This section also defines the complete transformation pipeline that will be applied inside each CV fold to prevent leakage.

**Pipeline order:**
1. Target transformation (train labels only)
2. Feature outlier handling (winsorize)
3. Missing value imputation
4. Feature creation (derived ratios, binary flags)
5. Encoding and final matrix alignment

---

### 4.1 Target Variable Transformation

Raw `return_pct` is extremely right-skewed (skew ≈ 38). A single stock with +10,571% return contributes more to squared-error loss than hundreds of typical stocks. You have three options:

**Option A — Clip at percentiles** *(recommended starting point)*  
Cap `return_pct` at e.g. the 99th percentile (~250%) for training. Pros: simple, retains interpretability. Cons: model cannot predict extreme returns.

**Option B — Signed log transform**  
Apply `y = np.sign(x) * np.log1p(np.abs(x))`. Pros: compresses the tail symmetrically. Cons: must invert before submission — easy to forget.

**Option C — No transformation**  
Let the model handle the raw target natively. Recommended only for LightGBM (tree models are less sensitive to target scale than linear ones).

**Start with Option A (clipped).** Also run Option C on LightGBM to compare. Always evaluate on the **original unclipped** validation targets.

In [ ]:
# 4.1 — Apply chosen target transformation to y_train; define inverse transform for submission


### 4.2 Feature Outlier Handling — Winsorization

Many financial ratios contain extreme values that are data errors or artifacts (e.g. `pe_ttm` max = 1.25 billion). These will dominate gradient computations in linear models and distort imputation statistics.

**Strategy: clip each feature at the 1st and 99th percentile of the training data.**

This preserves the direction of valid extreme values while preventing a handful of erroneous data points from dominating the model.

**Features requiring special attention:**
- `pe_ttm` — values > ~500 are almost certainly data artifacts; winsorize aggressively
- `price_to_book` — negative values are valid (company with negative stockholders' equity); preserve the sign
- `roe`, `roa` — extreme negative values from large losses are valid signals; use 1%/99% clipping, not zero-floor
- `debt_to_equity` — negative values are valid (more debt than equity); use 1%/99%
- Absolute-dollar columns (`revenue_ttm`, `total_assets`, etc.) — wide range is natural; consider log-scaling instead of clipping

Apply winsorization **before** imputation. Save the percentile thresholds computed on train and reuse them on test.

In [ ]:
# 4.2 — Winsorize each feature at the 1st and 99th percentile (fit on train only)


### 4.3 Missing Value Strategy

Apply the following column-by-column strategy. All fill values must be computed on **training data only**.

| Column | Missing % | Action |
|---|---|---|
| `dividends_paid_ttm` | 93% | **Drop** — almost entirely missing, no useful information |
| `dividend_yield` | 72% | Create `has_dividend` flag (1 = was not NaN), then fill with 0 |
| `gross_margin` | 62% | Fill with **median grouped by sector_code** |
| `growth_pe_ratio` | 35% | Fill with median by sector; or drop if permutation importance is near zero |
| `inventory`, `goodwill`, `current_assets/liabilities` | 30–50% | Fill with 0 (missing often means the line item genuinely doesn't exist for this company) |
| `shares_outstanding` | 35% | Fill with `shares_diluted` where available, else median by sector |
| All others (< 25% missing) | < 25% | Median imputation globally |

**For LightGBM:** you may skip imputation entirely and pass raw NaN values — LightGBM learns the optimal split direction for missing values. Compare imputed vs. non-imputed variants.

In [ ]:
# 4.3 — Impute missing values per the strategy above; fit statistics on train only


### 4.4 Feature Engineering

Create derived features grounded in financial theory — these can capture signal that raw features miss.

| New feature | Formula | Rationale |
|---|---|---|
| `earnings_yield` | `1 / pe_ttm` | More stable than P/E; handles negative P/E naturally; linked to expected return in factor models |
| `has_dividend` | `1 if dividend_yield was not NaN else 0` | Dividend payers vs. growth stocks is a strong categorical distinction |
| `asset_turnover` | `revenue_ttm / total_assets` | Operational efficiency; complements ROA |
| `leverage_ratio` | `total_assets / stockholders_equity` | Financial risk amplification |
| `quarter` | extracted from `period_start` | Earnings-season seasonality (Q4 reports may carry different signals than Q1) |

**Drop redundant columns to reduce noise:**
- `eps_basic` — nearly identical to `eps_diluted`; keep only `eps_diluted`
- `dividends_ttm` and `dividends_paid_ttm` — collinear with `dividend_yield`; keep yield only
- `current_ratio` and `quick_ratio` are correlated (~0.98) — consider keeping only `quick_ratio`

Apply all transformations identically to train and test.

In [ ]:
# 4.4 — Create derived features; drop redundant columns


### 4.5 Encode & Finalize Feature Matrix

**Categorical encoding — `sector_code`:**
- Treat as categorical (not ordinal) — one-hot encode into 11 dummy columns
- Alternative: leave as integer and use LightGBM's native `categorical_feature` — cleaner and often better

**Date and metadata:**
- `start_year`: keep as a numeric feature — it captures macro regime (COVID, rate hikes)
- `quarter`: extract from `period_start` as 1–4 (available in train only; compute from context for test if needed using `start_year`)
- Drop from feature matrix: `id`, `ticker`, `period_start`, `period_end`, `return_pct`

**Final check:**
- Confirm `X_train.columns` equals `X_test.columns` — any mismatch will cause silent errors
- No NaN values should remain in the final matrix (unless feeding directly to LightGBM)
- Print the final feature count and a quick `.describe()` to confirm all values are in reasonable ranges

In [ ]:
# 4.5 — One-hot encode sector_code; drop metadata columns; align train and test columns


---
## Section 5 — Validation Strategy

### Why standard K-Fold is wrong here

This is a **panel dataset** — 1,734 tickers × up to 4 quarterly snapshots × 4 years. A random 80/20 split will put different quarters of the **same ticker** in both train and validation. The model will implicitly memorize ticker-specific patterns (e.g. Apple's typical P/E range) rather than learning generalizable fundamentals. This inflates CV scores without reflecting real-world performance.

### Time-based cross-validation with GroupKFold

Use `GroupKFold(n_splits=4)` with `groups = train['start_year']`. This guarantees each fold validates on **one complete year** of data the model never trained on:

| Fold | Training years | Validation year |
|---|---|---|
| 1 | 2020, 2021, 2022 | 2019 |
| 2 | 2019, 2021, 2022 | 2020 (COVID) |
| 3 | 2019, 2020, 2022 | 2021 (post-COVID boom) |
| 4 | 2019, 2020, 2021 | 2022 (rate-hike bear market) |

**Leakage checklist:**
- Imputation statistics (medians) must be computed on the training fold only
- Winsorization thresholds must be computed on the training fold only
- `StandardScaler` must be fit on training fold only
- `start_year` can be used as a feature (it's available in test), but watch for regime overfitting

In [ ]:
# 5 — Set up GroupKFold CV splitter and define the score_model() helper function


---
## Section 6 — Baseline Models

We train four models of increasing complexity, each using the same time-aware CV from Section 5. The goal is not to find the final model here — it is to establish honest performance benchmarks and understand which model family fits this problem best.

> A model that meaningfully beats the naive baseline on Spearman ρ already has real-world utility for stock selection.

---

### 6.1 Naive Baseline — Predict the Mean

The simplest possible model: predict the training-fold mean `return_pct` for every stock in the validation fold.

This sets an absolute floor. Any real model must beat this on MAE. Spearman ρ will be exactly 0.0 (no ranking information).

**Implement** using the `score_model` helper from Section 7.1. Record MAE, RMSE, R², and Spearman ρ.

In [ ]:
# 6.1 — Compute naive baseline (predict mean) and record scores


### 6.2 Regularized Linear Regression — Ridge & Lasso

A simple, interpretable model. Reveals which fundamentals have a **linear** relationship with returns, and quantifies the magnitude.

**Setup:**
- StandardScale all numeric features **on the training fold only**, then apply the same scaler to the validation fold
- Tune `alpha` (regularization strength): try `[0.01, 0.1, 1, 10, 100]` via the same GroupKFold CV

**What to report:**
- CV MAE and RMSE for Ridge and Lasso
- Top positive and negative Lasso coefficients — e.g. is a lower P/E associated with higher returns? Higher revenue growth?
- How many features does Lasso zero out? (shows which fundamentals carry no linear signal)

**Expected performance:** noticeably worse than tree models — stock returns are not linearly separable from fundamentals. But this is an important sanity check and provides an interpretable lower bound for comparison.

In [ ]:
# 6.2 — Fit Ridge and Lasso; tune alpha via CV; plot coefficients


### 6.3 Random Forest

A strong non-linear baseline that captures feature interactions and is robust to feature-level outliers.

**Key advantages over linear models:**
- Captures non-linear relationships between fundamentals and returns
- Feature importance is easy to extract and interpret
- Robust to irrelevant features (they simply don't get selected for splits)

**Recommended starting hyperparameters:**
```
n_estimators    : 300
max_depth       : 15        (limit depth to avoid memorizing extreme target values)
min_samples_leaf: 30        (higher = more regularization; important with noisy targets)
max_features    : 'sqrt'    (standard; reduces correlation between trees)
```

**Limitations to be aware of:**
- Cannot extrapolate beyond the training target range (predictions are bounded by min/max of training returns)
- Cannot handle NaN values natively — imputation from Section 4.3 is required before fitting
- Slower to train than LightGBM at scale

In [ ]:
# 6.3 — Fit Random Forest with CV; record scores and OOF predictions


### 6.4 LightGBM

The strongest baseline for tabular data — gradient boosted trees with native support for missing values and categorical features.

**Why it tends to win on structured data:**
- Handles NaN values natively (no imputation required)
- `categorical_feature` parameter handles `sector_code` without one-hot encoding
- Leaf-wise tree growth finds complex feature interactions efficiently

**Recommended starting hyperparameters:**
```
n_estimators     : 1000  (use early stopping to find optimal)
learning_rate    : 0.05
num_leaves       : 63
min_child_samples: 50   (regularizes against overfitting small noisy leaves)
subsample        : 0.8
colsample_bytree : 0.8
reg_alpha        : 0.1  (L1)
reg_lambda       : 1.0  (L2)
```

**Compare two variants:**
1. Trained on **clipped** target (from 4.1 Option A) — better calibration on typical stocks
2. Trained on **raw** target (Option C) — may score better if extreme returns appear in the test set

**Tip:** Use the first CV fold to tune `n_estimators` via early stopping, then fix it for the full CV run.

In [ ]:
# 6.4 — Fit LightGBM with early stopping; record CV scores


---
## Section 7 — Evaluation

### 7.1 Metrics

Four complementary metrics give a full picture of model quality:

| Metric | What it measures | When to use |
|---|---|---|
| **MAE** | Average absolute error in % return | Primary metric — robust to the extreme-return tail |
| **RMSE** | Error penalizing large mistakes more | Sensitive to outliers; watch it but don't optimize for it |
| **R²** | Fraction of variance explained | Useful context but misleading with skewed targets |
| **Spearman ρ** | Rank correlation — does the model correctly order stocks? | Critical for portfolio applications; even 0.10 is useful |

**Scoring function to implement:**  
A helper `score_model(model, X, y, groups)` that runs `GroupKFold(n_splits=4)` on `start_year` and returns `[MAE, RMSE, R², Spearman ρ]` averaged across folds. This function will be reused for every model in Section 6.

In [ ]:
# 7.1 — Define scoring helper function; run all models through CV and record results


### 7.2 Cross-Validated Model Comparison

Aggregate out-of-fold scores from Section 6 and compare all baselines side by side.

| Model | CV MAE | CV RMSE | CV R² | CV Spearman ρ |
|---|---|---|---|---|
| Naive (predict mean) | — | — | — | — |
| Ridge | — | — | — | — |
| Lasso | — | — | — | — |
| Random Forest | — | — | — | — |
| LightGBM | — | — | — | — |

**Interpretation notes:**
- **MAE** is the primary metric — less sensitive to the extreme-return tail than RMSE
- **Spearman ρ** near 0 means the model cannot rank stocks; even ρ = 0.1–0.15 is meaningful for real stock selection
- If LightGBM with no target transformation is not clearly better than Random Forest, revisit the preprocessing pipeline
- A high RMSE with low MAE signals the model is being dragged by a few extreme predictions

In [ ]:
# 7.2 — Collect OOF scores for all models and fill in the comparison table


### 7.3 Feature Importance & SHAP Analysis

Apply to the best-performing model from 7.2.

**Built-in importance (tree models):**
- Plot top 20 features by gain importance
- Caveat: this favors high-cardinality / continuous features and can overstate their importance

**Permutation importance (model-agnostic, more reliable):**
- Shuffle each feature on the validation fold and measure the drop in MAE
- Features with no drop have no signal — candidates for removal to simplify the model

**SHAP analysis** (`pip install shap`):
- **Beeswarm plot** — shows direction and magnitude of each feature's effect across all samples
- **Dependence plot** for top 4–5 features (e.g. `pe_ttm`, `roe`, `revenue_growth_yoy`, `price_to_book`) — shows the relationship between feature value and SHAP value
- **Key question to answer with SHAP:** does low P/E reliably predict higher returns? Does high ROE? The answers will guide feature engineering decisions.

In [ ]:
# 7.3 — Feature importance (permutation) and SHAP beeswarm + dependence plots


### 7.4 Error Analysis

Diagnose *where* and *why* the best model fails — this directly guides what to improve next.

**Recommended slices:**

| Slice | What to look for |
|---|---|
| Residuals vs. predicted value | Heteroskedasticity — errors grow for large predicted returns |
| Residuals by `sector_code` | Sectors that are systematically over- or under-predicted |
| Residuals by `start_year` | Worse on certain market regimes (2020 COVID, 2022 rate hikes)? |
| Errors by return bucket | Create buckets: < −50%, −50–0%, 0–50%, 50–200%, > 200% |

**Expected finding:** tree models will systematically under-predict extreme movers (returns > 200%) because they average within leaves. If extreme returns dominate the leaderboard metric, you may need a two-stage model or a separate model for the tail.

**Plot:** scatter of actual vs. predicted with a 45° reference line. Points far from the line are systematic failures.

In [ ]:
# 7.4 — Error analysis: residuals by sector, year, and return magnitude bucket


### 7.5 Submission

Generate predictions on the test set and format for submission.

**Steps:**
1. Apply the exact same preprocessing pipeline to `test` — using thresholds and imputation values fitted on `train` only
2. Predict with the best cross-validated model
3. If you trained on a transformed target (clipped or log), invert the transformation before saving
4. Clip predictions to a sensible range (e.g. −99 to +500) to avoid nonsensical outputs
5. Format as `id`, `return_pct` and save to CSV

**Sanity checks before submitting:**
- Compare the distribution of test predictions to out-of-fold train predictions — they should be similar
- Check for any NaN or infinite predictions
- Verify `id` alignment with `sample_submission.csv`

In [ ]:
# 7.5 — Generate predictions on test set and save submission CSV
